# 03 - Análisis Principal

Este notebook desarrolla el análisis principal del proyecto GESAssist.

Pregunta de análisis:

**¿Qué patrones se observan en los diagnósticos clasificados como GES y No GES considerando el diagnóstico y la edad del paciente?**


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/dataset_limpio.csv")
df.head()

## Métricas generales

In [ ]:
total_casos = len(df)
casos_ges = int(df["ges"].sum())
casos_no_ges = total_casos - casos_ges
porcentaje_ges = casos_ges / total_casos * 100

print("Total de casos:", total_casos)
print("Casos GES:", casos_ges)
print("Casos No GES:", casos_no_ges)
print("Porcentaje GES:", round(porcentaje_ges, 2), "%")

## Proporción de casos GES y No GES

In [ ]:
df_clasificacion = df["ges"].map({True: "GES", False: "No GES"}).value_counts().reset_index()
df_clasificacion.columns = ["clasificacion", "cantidad"]

fig = px.pie(
    df_clasificacion,
    names="clasificacion",
    values="cantidad",
    title="Proporción de casos GES y No GES"
)
fig.show()

## Diagnósticos GES más frecuentes

In [ ]:
diagnosticos_ges = (
    df[df["ges"] == True]["diagnostic"]
    .value_counts()
    .head(15)
    .reset_index()
)

diagnosticos_ges.columns = ["diagnostic", "cantidad"]
diagnosticos_ges

In [ ]:
fig = px.bar(
    diagnosticos_ges,
    x="cantidad",
    y="diagnostic",
    orientation="h",
    text="cantidad",
    title="Top 15 diagnósticos GES más frecuentes"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Diagnósticos No GES más frecuentes

In [ ]:
diagnosticos_no_ges = (
    df[df["ges"] == False]["diagnostic"]
    .value_counts()
    .head(15)
    .reset_index()
)

diagnosticos_no_ges.columns = ["diagnostic", "cantidad"]
diagnosticos_no_ges

In [ ]:
fig = px.bar(
    diagnosticos_no_ges,
    x="cantidad",
    y="diagnostic",
    orientation="h",
    text="cantidad",
    title="Top 15 diagnósticos No GES más frecuentes"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Análisis por grupo etario

In [ ]:
df["grupo_edad"] = pd.cut(
    df["age"],
    bins=[0, 18, 30, 45, 60, 75, 100],
    labels=["0-18", "19-30", "31-45", "46-60", "61-75", "76+"]
)

grupo_edad = (
    df.groupby(["grupo_edad", "ges"], observed=False)
    .size()
    .reset_index(name="cantidad")
)

grupo_edad["clasificacion"] = grupo_edad["ges"].map({True: "GES", False: "No GES"})
grupo_edad

In [ ]:
fig = px.bar(
    grupo_edad,
    x="grupo_edad",
    y="cantidad",
    color="clasificacion",
    barmode="group",
    title="Casos GES y No GES por grupo etario",
    labels={"grupo_edad": "Grupo de edad", "cantidad": "Cantidad"}
)
fig.show()

## Hallazgo principal

El dataset presenta una mayor cantidad de casos No GES que GES. Sin embargo, dentro de los casos GES se observan diagnósticos recurrentes como catarata, cáncer de mama, cáncer de colon, insuficiencia renal crónica y otros diagnósticos específicos.

Un hallazgo relevante es que la clasificación GES no depende únicamente del texto del diagnóstico. También puede estar relacionada con la edad del paciente y con criterios específicos de cobertura.

Este análisis permite construir un dashboard que resume la información y facilita la exploración por diagnóstico, edad y clasificación.
